# Can we predict late delivery?
Target: `late_delivery`. Prediction is made at dispatch, for future consignments, so the split is temporal.

Decisions up front (checklist.md):
- `consignment` is an identifier (unique per row) -> dropped.
- `claim_paid_eur` is recorded after delivery (a claim settled for the consignment) and is a near-proxy for the outcome -> **excluded as leakage**.
- Features known at dispatch: `carrier`, `distance_km`, `weight_kg`, `parcels`, hour-of-day and day-of-week of `dispatched_on`. Calendar month is not used (cannot generalise to later rows).
- Test set = latest 20% of rows by dispatch time, untouched until the final step.
- Cross-validation: expanding-window `TimeSeriesSplit` inside the training rows only; preprocessing is inside a Pipeline so it is fit per fold.

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, brier_score_loss
from sklearn.base import clone

df=pd.read_csv('data.csv',parse_dates=['dispatched_on']).sort_values('dispatched_on').reset_index(drop=True)
df['hour']=df.dispatched_on.dt.hour; df['dow']=df.dispatched_on.dt.dayofweek
print(df.shape, df.consignment.is_unique, df.late_delivery.mean())

## 1. Test split (latest 20% by time)

In [ ]:
cut=int(len(df)*0.8)
train,test=df.iloc[:cut].copy(),df.iloc[cut:].copy()
print(train.dispatched_on.min(),train.dispatched_on.max()); print(test.dispatched_on.min(),test.dispatched_on.max())
print('train late rate',train.late_delivery.mean(),'| test rows',len(test))

Only the training rows are examined from here on. Late rate by month in training data, to understand drift:

In [ ]:
print(train.groupby(train.dispatched_on.dt.to_period('M')).late_delivery.agg(['mean','size']).round(3).T.to_string())
print(train.groupby('carrier').late_delivery.mean())

## 2. Candidate models (named before scoring)
1. `baseline`: predicts the training prior (trivial).
2. `logreg`: logistic regression on scaled numeric + one-hot carrier + hour/dow.
3. `hgb_d2`, `hgb_d3`: histogram gradient boosting, same features, max_depth 2 or 3, learning_rate 0.05.

Metric for selection: mean ROC AUC across CV folds (threshold-free); average precision as secondary. Decision threshold chosen on pooled out-of-fold predictions of the chosen model, maximising F1.

In [ ]:
num=['distance_km','weight_kg','parcels','hour','dow']; cat=['carrier']; feats=num+cat
def pre(scale):
    return ColumnTransformer([('n',StandardScaler() if scale else 'passthrough',num),('c',OneHotEncoder(handle_unknown='ignore'),cat)])
models={
 'baseline':make_pipeline(pre(False),DummyClassifier(strategy='prior')),
 'logreg':make_pipeline(pre(True),LogisticRegression(max_iter=1000)),
 'hgb_d2':make_pipeline(pre(False),HistGradientBoostingClassifier(max_depth=2,learning_rate=0.05,random_state=0)),
 'hgb_d3':make_pipeline(pre(False),HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,random_state=0)),
}
X,y=train[feats],train.late_delivery
tscv=TimeSeriesSplit(n_splits=5)
rows=[];oof={}
for n,m in models.items():
    aucs=[];aps=[];p=np.full(len(train),np.nan)
    for tr,va in tscv.split(X):
        f=clone(m).fit(X.iloc[tr],y.iloc[tr]); pr=f.predict_proba(X.iloc[va])[:,1]; p[va]=pr
        aucs.append(roc_auc_score(y.iloc[va],pr)); aps.append(average_precision_score(y.iloc[va],pr))
    oof[n]=p; rows.append((n,np.mean(aucs),np.std(aucs),np.mean(aps)))
cv=pd.DataFrame(rows,columns=['model','auc_mean','auc_std','ap_mean']).round(4); print(cv)

## 3. Choice (CV only)

In [ ]:
best=cv[cv.model!='baseline'].sort_values('auc_mean',ascending=False).iloc[0].model
mask=~np.isnan(oof[best]); yo,po=y[mask],oof[best][mask]
ths=np.linspace(0.05,0.95,91); f1s=[f1_score(yo,po>=t) for t in ths]
thr=float(ths[int(np.argmax(f1s))])
print('chosen model:',best,'| threshold:',round(thr,2),'| OOF F1:',round(max(f1s),3),'| OOF F1 at all-positive:',round(f1_score(yo,np.ones(len(yo))),3))

**Written choice (made before test scoring):** the model printed above, with the features listed at the top and the threshold above, all selected on cross-validation only. The test set is scored once below.

## 4. Refit on all training rows, score test once

In [ ]:
final=clone(models[best]).fit(X,y); base=clone(models['baseline']).fit(X,y)
Xt,yt=test[feats],test.late_delivery
for n,m in [('baseline',base),(best,final)]:
    p=m.predict_proba(Xt)[:,1]
    print(n,'AUC',round(roc_auc_score(yt,p),3),'AP',round(average_precision_score(yt,p),3),'Brier',round(brier_score_loss(yt,p),4))
pt=final.predict_proba(Xt)[:,1]
print('F1 at chosen threshold',round(f1_score(yt,pt>=thr),3),'| test late rate',round(yt.mean(),4),'| mean predicted',round(pt.mean(),3))

## 5. Interpretation
See the outputs above for numbers. Caveats: the late rate varies strongly over time (see monthly table) and is far lower in the latest rows than in training, so calibration and threshold-based metrics on the test set are affected by that shift; test AUC measures ranking ability only. With few late cases in the test window, metrics are noisy. `claim_paid_eur` was deliberately excluded; including it would give near-perfect but unusable predictions.